# CFD Data Processing - Part 3: Excel Outputs

This notebook loads processed data from Part 1 and generates Excel files with formatted tables.
Uses optimized trim values from Part 2 if available, otherwise uses fixed iterations.

Creates sheets: Data Summary, Turbulence_Comparison, Coefficients.

## Setup

In [10]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
import pickle
import time
import glob
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

## Load Processed Data

In [11]:
# Automatically find the pickle file generated by Part 1
# This searches for processed_data.pkl starting from user's home directory

# Start search from a higher directory to find the Thesis folder
# Try multiple search locations to find the pickle file
search_roots = [
    r"C:\Users\lukek\OneDrive\Documents\Thesis",  # Direct path to Thesis folder
    os.path.expanduser("~"),  # User home directory
]

pickle_files = []
for root in search_roots:
    if os.path.exists(root):
        search_pattern = os.path.join(root, "**", "processed_data.pkl")
        found = glob.glob(search_pattern, recursive=True)
        pickle_files.extend(found)
        if found:
            break  # Stop searching once we find files

# Filter to get the most recently modified pickle file
if pickle_files:
    pickle_file = max(pickle_files, key=os.path.getmtime)
    print(f"✓ Found pickle file: {pickle_file}")
    print(f"  Last modified: {time.ctime(os.path.getmtime(pickle_file))}")
else:
    print("\n❌ ERROR: No processed_data.pkl file found!")
    print("\n💡 Solution:")
    print("   Run notebook 1_data_processing.ipynb first to generate processed_data.pkl")
    raise FileNotFoundError(f"Run Part 1 first to create processed_data.pkl")

with open(pickle_file, 'rb') as f:
    data_package = pickle.load(f)

all_data = data_package['all_data']
config_info = data_package['config_info']
paths = data_package['paths']

POSITION_MAP = config_info['POSITION_MAP']
VALUE_MAPPINGS = config_info['VALUE_MAPPINGS']
TURBULENCE_ORDER = config_info['TURBULENCE_ORDER']
COMPARISON_CONFIGS = config_info['COMPARISON_CONFIGS']
CONFIG_EXTRACTION_METHOD = config_info['CONFIG_EXTRACTION_METHOD']

base_path = paths['base_path']
output_dir = paths['output_dir']

print(f"\n✓ Data loaded successfully!")
print(f"  Configurations: {len(set(config for config, aoa in all_data.keys()))}")
print(f"  Total combinations: {len(all_data)}")

# Load convergence results if available
convergence_pickle = os.path.join(output_dir, 'convergence_results.pkl')
if os.path.exists(convergence_pickle):
    with open(convergence_pickle, 'rb') as f:
        conv_package = pickle.load(f)
        convergence_results = conv_package['convergence_results']
        RUN_CONVERGENCE_ANALYSIS = True
        num_iterations = conv_package.get('num_iterations', 150)
        print(f"✓ Convergence results loaded: {len(convergence_results)} configurations")
else:
    convergence_results = {}
    RUN_CONVERGENCE_ANALYSIS = False
    num_iterations = 150
    print("⚠ No convergence results - using fixed iterations (150)")

✓ Found pickle file: C:\Users\lukek\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\processed_data.pkl
  Last modified: Sun Dec  7 15:04:57 2025

✓ Data loaded successfully!
  Configurations: 16
  Total combinations: 16
✓ Convergence results loaded: 16 configurations


## Helper Functions

In [12]:
def extract_aoa_number(aoa_string):
    """Extract numeric AoA value from string like 'AoA_5' -> 5"""
    try:
        return int(aoa_string.split('_')[1])
    except:
        return 0

def get_optimized_data(config, aoa, data, convergence_results):
    """Get optimized data if convergence analysis was run, otherwise use fixed iterations"""
    if RUN_CONVERGENCE_ANALYSIS and (config, aoa) in convergence_results:
        conv = convergence_results[(config, aoa)]
        lift_min_cov_idx = np.argmin(conv['lift']['cov'])
        drag_min_cov_idx = np.argmin(conv['drag']['cov'])
        
        optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
        optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
        optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
        
        lift_values = data['lift'][optimal_trim:]
        drag_values = data['drag'][optimal_trim:]
    else:
        lift_values = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_values = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    return lift_values, drag_values

## User Inputs (Optional Override)

In [13]:
# Uncomment to override settings from pickle file
# COMPARISON_CONFIGS = {'SST': '4.3.1.2', 'RNG': '4.3.2.1', 'RSM': '4.3.3.1'}
# TURBULENCE_ORDER = {'k-epsilon': 1, 'k-omega': 2, 'Transition SST': 3}

# Coefficient calculation parameters
SPAN = 0.85344          # meters
CHORD = 0.1             # meters
AIR_DENSITY = 1.225     # kg/m^3
VELOCITY = 24.38        # m/s

REFERENCE_AREA = SPAN * CHORD
DYNAMIC_PRESSURE = 0.5 * AIR_DENSITY * VELOCITY**2
Q_times_A = DYNAMIC_PRESSURE * REFERENCE_AREA

print(f"Reference Area = {REFERENCE_AREA:.6f} m²")
print(f"Dynamic Pressure = {DYNAMIC_PRESSURE:.4f} Pa")
print(f"Q × A = {Q_times_A:.4f}")

Reference Area = 0.085344 m²
Dynamic Pressure = 364.0604 Pa
Q × A = 31.0704


## Create Data Summary Sheet

In [14]:
# Create Excel file
excel_file = os.path.join(output_dir, "SUMMARY_Statistics.xlsx")

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
group_header_font = Font(name='Calibri', size=12, bold=True, color='FFFFFF')
group_header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
data_alignment = Alignment(horizontal='center', vertical='center')
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

# Group data by base configuration
base_config_data = defaultdict(list)
for (config, aoa), data in all_data.items():
    config_parts = config.split('.')
    if len(config_parts) > 1:
        base_config = '.'.join(config_parts[:-1])
    else:
        base_config = config
    
    lift_values, drag_values = get_optimized_data(config, aoa, data, convergence_results)
    
    lift_mean = np.mean(lift_values) if lift_values else 0
    lift_std = np.std(lift_values) if lift_values else 0
    lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
    
    drag_mean = np.mean(drag_values) if drag_values else 0
    drag_std = np.std(drag_values) if drag_values else 0
    drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
    
    base_config_data[base_config].append({
        'aoa': aoa,
        'aoa_num': extract_aoa_number(aoa),
        'turbulence_model': data['turbulence_model'],
        'num_points': len(lift_values),
        'lift_mean': lift_mean,
        'lift_cov': lift_cov,
        'drag_mean': drag_mean,
        'drag_cov': drag_cov
    })

# Sort each group by AoA
for base_config in base_config_data:
    base_config_data[base_config].sort(key=lambda x: x['aoa_num'])

# Sort base configs
def get_base_config_sort_key(base_config):
    if base_config in base_config_data and base_config_data[base_config]:
        turb_model = base_config_data[base_config][0]['turbulence_model']
        return (TURBULENCE_ORDER.get(turb_model, 99), base_config)
    return (99, base_config)

sorted_base_configs = sorted(base_config_data.keys(), key=get_base_config_sort_key)

# Create workbook
wb = Workbook()
ws = wb.active
ws.title = 'Data Summary'

columns = ['Turbulence Model', 'Angle of Attack', 'Lift Mean (N)', 'Lift COV (%)', 'Drag Mean (N)', 'Drag COV (%)', 'Num Points']
current_row = 1

for base_config in sorted_base_configs:
    data_rows = base_config_data[base_config]
    if not data_rows:
        continue
    
    # Section header
    ws.cell(row=current_row, column=1, value=base_config)
    ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(columns))
    header_cell = ws.cell(row=current_row, column=1)
    header_cell.font = group_header_font
    header_cell.fill = group_header_fill
    header_cell.alignment = Alignment(horizontal='center', vertical='center')
    header_cell.border = border_style
    ws.row_dimensions[current_row].height = 25
    current_row += 1
    
    # Column headers
    for col_idx, col_name in enumerate(columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    ws.row_dimensions[current_row].height = 25
    current_row += 1
    
    # Data rows
    for row_idx, row_data in enumerate(data_rows):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        
        values = [
            row_data['turbulence_model'],
            row_data['aoa'],
            f"{row_data['lift_mean']:.1f}",
            f"{row_data['lift_cov']:.1f}",
            f"{row_data['drag_mean']:.1f}",
            f"{row_data['drag_cov']:.1f}",
            row_data['num_points']
        ]
        
        for col_idx, val in enumerate(values, 1):
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
        
        current_row += 1
    
    current_row += 1  # Blank row

# Autofit columns
for col_idx in range(1, len(columns) + 1):
    max_length = 0
    column_letter = get_column_letter(col_idx)
    for row in ws.iter_rows(min_col=col_idx, max_col=col_idx):
        for cell in row:
            try:
                if cell.value and len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
    adjusted_width = min(max_length + 2, 25)
    ws.column_dimensions[column_letter].width = adjusted_width

wb.save(excel_file)

print(f"✓ Excel file created: {excel_file}")
print(f"✓ Sheet: Data Summary")
print(f"✓ {len(sorted_base_configs)} configurations with grouped AoAs")

✓ Excel file created: C:\Users\lukek\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\SUMMARY_Statistics.xlsx
✓ Sheet: Data Summary
✓ 1 configurations with grouped AoAs


## Create Turbulence Comparison Sheet

In [15]:
print("=" * 70)
print("TURBULENCE MODEL COMPARISON")
print("=" * 70)

active_configs = {k: v for k, v in COMPARISON_CONFIGS.items() if v and str(v).strip()}
print("\nConfigurations being compared:")
for model, config in active_configs.items():
    print(f"  {model}: {config}")

if not active_configs:
    print("⚠ No configurations specified in COMPARISON_CONFIGS!")
else:
    # Get all unique AoAs
    all_aoas = set()
    for (config, aoa), data in all_data.items():
        parts = config.split('.')
        if len(parts) >= 4:
            base_config = '.'.join(parts[:4])
            if base_config in active_configs.values():
                all_aoas.add(aoa)
    
    sorted_aoas = sorted(all_aoas, key=extract_aoa_number)
    model_order = {'SST': 1, 'RNG': 2, 'RSM': 3, 'k-epsilon': 4}
    sorted_models = sorted(active_configs.keys(), key=lambda x: model_order.get(x, 99))
    
    print(f"\nAoAs found: {sorted_aoas}")
    print(f"Models: {sorted_models}")
    
    # Collect data
    lift_data = {}
    drag_data = {}
    lift_cov_data = {}
    drag_cov_data = {}
    
    for aoa in sorted_aoas:
        lift_data[aoa] = {}
        drag_data[aoa] = {}
        lift_cov_data[aoa] = {}
        drag_cov_data[aoa] = {}
        
        for model in sorted_models:
            base_config = active_configs[model]
            model_lift_values = []
            model_drag_values = []
            
            for (config, config_aoa), data in all_data.items():
                if config_aoa == aoa:
                    parts = config.split('.')
                    if len(parts) >= 4:
                        this_base = '.'.join(parts[:4])
                        if this_base == base_config:
                            lift_values, drag_values = get_optimized_data(config, aoa, data, convergence_results)
                            if lift_values:
                                model_lift_values.extend(lift_values)
                            if drag_values:
                                model_drag_values.extend(drag_values)
            
            lift_data[aoa][model] = np.mean(model_lift_values) if model_lift_values else None
            drag_data[aoa][model] = np.mean(model_drag_values) if model_drag_values else None
            
            if model_lift_values:
                lift_mean = np.mean(model_lift_values)
                lift_std = np.std(model_lift_values)
                lift_cov_data[aoa][model] = (lift_std / lift_mean * 100) if lift_mean != 0 else None
            else:
                lift_cov_data[aoa][model] = None
                
            if model_drag_values:
                drag_mean = np.mean(model_drag_values)
                drag_std = np.std(model_drag_values)
                drag_cov_data[aoa][model] = (drag_std / drag_mean * 100) if drag_mean != 0 else None
            else:
                drag_cov_data[aoa][model] = None
    
    # Create dataframes
    lift_rows = []
    drag_rows = []
    lift_cov_rows = []
    drag_cov_rows = []
    
    for aoa in sorted_aoas:
        sst_lift = lift_data[aoa].get('SST')
        sst_drag = drag_data[aoa].get('SST')
        sst_lift_cov = lift_cov_data[aoa].get('SST')
        sst_drag_cov = drag_cov_data[aoa].get('SST')
        
        lift_row = {'AoA': aoa}
        drag_row = {'AoA': aoa}
        lift_cov_row = {'AoA': aoa}
        drag_cov_row = {'AoA': aoa}
        
        for model in sorted_models:
            # Lift
            val = lift_data[aoa].get(model)
            if val is not None:
                lift_row[f'{model} (N)'] = round(val, 1)
                if model != 'SST' and sst_lift is not None and sst_lift != 0:
                    lift_row[f'{model} %Diff'] = round(((val - sst_lift) / sst_lift * 100), 1)
                elif model != 'SST':
                    lift_row[f'{model} %Diff'] = 'N/A'
            else:
                lift_row[f'{model} (N)'] = 'N/A'
                if model != 'SST':
                    lift_row[f'{model} %Diff'] = 'N/A'
            
            # Drag
            val = drag_data[aoa].get(model)
            if val is not None:
                drag_row[f'{model} (N)'] = round(val, 1)
                if model != 'SST' and sst_drag is not None and sst_drag != 0:
                    drag_row[f'{model} %Diff'] = round(((val - sst_drag) / sst_drag * 100), 1)
                elif model != 'SST':
                    drag_row[f'{model} %Diff'] = 'N/A'
            else:
                drag_row[f'{model} (N)'] = 'N/A'
                if model != 'SST':
                    drag_row[f'{model} %Diff'] = 'N/A'
            
            # Lift COV
            val = lift_cov_data[aoa].get(model)
            if val is not None:
                lift_cov_row[f'{model} COV (%)'] = round(val, 1)
                if model != 'SST' and sst_lift_cov is not None and sst_lift_cov != 0:
                    lift_cov_row[f'{model} COV %Diff'] = round(((val - sst_lift_cov) / sst_lift_cov * 100), 1)
                elif model != 'SST':
                    lift_cov_row[f'{model} COV %Diff'] = 'N/A'
            else:
                lift_cov_row[f'{model} COV (%)'] = 'N/A'
                if model != 'SST':
                    lift_cov_row[f'{model} COV %Diff'] = 'N/A'
            
            # Drag COV
            val = drag_cov_data[aoa].get(model)
            if val is not None:
                drag_cov_row[f'{model} COV (%)'] = round(val, 1)
                if model != 'SST' and sst_drag_cov is not None and sst_drag_cov != 0:
                    drag_cov_row[f'{model} COV %Diff'] = round(((val - sst_drag_cov) / sst_drag_cov * 100), 1)
                elif model != 'SST':
                    drag_cov_row[f'{model} COV %Diff'] = 'N/A'
            else:
                drag_cov_row[f'{model} COV (%)'] = 'N/A'
                if model != 'SST':
                    drag_cov_row[f'{model} COV %Diff'] = 'N/A'
        
        lift_rows.append(lift_row)
        drag_rows.append(drag_row)
        lift_cov_rows.append(lift_cov_row)
        drag_cov_rows.append(drag_cov_row)
    
    lift_df = pd.DataFrame(lift_rows)
    drag_df = pd.DataFrame(drag_rows)
    lift_cov_df = pd.DataFrame(lift_cov_rows)
    drag_cov_df = pd.DataFrame(drag_cov_rows)
    
    # Write to Excel
    try:
        wb_existing = load_workbook(excel_file)
        if 'Turbulence_Comparison' in wb_existing.sheetnames:
            del wb_existing['Turbulence_Comparison']
            wb_existing.save(excel_file)
        wb_existing.close()
    except:
        pass
    
    time.sleep(0.3)
    
    wb = load_workbook(excel_file)
    ws = wb.create_sheet('Turbulence_Comparison')
    
    section_font = Font(name='Calibri', size=12, bold=True, color='366092')
    diff_font = Font(name='Calibri', size=11, color='4169E1')
    
    current_row = 1
    
    # LIFT SECTION
    ws.cell(row=current_row, column=1, value="LIFT COMPARISON (N)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    for col_idx, col_name in enumerate(lift_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    for row_idx, row_data in lift_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(lift_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    current_row += 1
    
    # DRAG SECTION
    ws.cell(row=current_row, column=1, value="DRAG COMPARISON (N)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    for col_idx, col_name in enumerate(drag_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    for row_idx, row_data in drag_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(drag_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    current_row += 2
    
    # LIFT COV SECTION
    ws.cell(row=current_row, column=1, value="LIFT COV COMPARISON (%)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    for col_idx, col_name in enumerate(lift_cov_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    for row_idx, row_data in lift_cov_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(lift_cov_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    current_row += 1
    
    # DRAG COV SECTION
    ws.cell(row=current_row, column=1, value="DRAG COV COMPARISON (%)")
    ws.cell(row=current_row, column=1).font = section_font
    current_row += 1
    
    for col_idx, col_name in enumerate(drag_cov_df.columns, 1):
        cell = ws.cell(row=current_row, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    current_row += 1
    
    for row_idx, row_data in drag_cov_df.iterrows():
        fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
        for col_idx, col_name in enumerate(drag_cov_df.columns, 1):
            val = row_data[col_name]
            cell = ws.cell(row=current_row, column=col_idx, value=val)
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
            if '%Diff' in col_name:
                cell.font = diff_font
        current_row += 1
    
    current_row += 2
    
    # Config note
    ws.cell(row=current_row, column=1, value="Configurations compared:")
    ws.cell(row=current_row, column=1).font = Font(name='Calibri', size=10, italic=True)
    current_row += 1
    
    for model, config in active_configs.items():
        ws.cell(row=current_row, column=1, value=f"  {model}: {config}")
        ws.cell(row=current_row, column=1).font = Font(name='Calibri', size=10, italic=True)
        current_row += 1
    
    # Autofit columns
    max_cols = max(len(lift_df.columns), len(drag_df.columns), len(lift_cov_df.columns), len(drag_cov_df.columns))
    for col_idx in range(1, max_cols + 1):
        max_length = 0
        column_letter = get_column_letter(col_idx)
        for row in ws.iter_rows(min_col=col_idx, max_col=col_idx):
            for cell in row:
                try:
                    if cell.value and len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
        adjusted_width = min(max_length + 2, 18)
        ws.column_dimensions[column_letter].width = adjusted_width
    
    wb.save(excel_file)
    wb.close()
    
    print(f"\n✓ Turbulence Comparison created with 4 tables")

TURBULENCE MODEL COMPARISON

Configurations being compared:
  4.3.NG: ['4.3.1.NG', '4.3.2.NG', '4.3.3.NG']

AoAs found: []
Models: ['4.3.NG']

✓ Turbulence Comparison created with 4 tables

✓ Turbulence Comparison created with 4 tables


## Create Coefficients Sheet

In [16]:
# Calculate coefficients
coefficient_data = {}

for (config, aoa), data in all_data.items():
    lift_values, drag_values = get_optimized_data(config, aoa, data, convergence_results)
    
    lift_mean = np.mean(lift_values) if lift_values else 0
    drag_mean = np.mean(drag_values) if drag_values else 0
    
    C_L = lift_mean / Q_times_A if Q_times_A != 0 else 0
    C_D = drag_mean / Q_times_A if Q_times_A != 0 else 0
    
    lift_std = np.std(lift_values) if lift_values else 0
    drag_std = np.std(drag_values) if drag_values else 0
    
    C_L_std = lift_std / Q_times_A if Q_times_A != 0 else 0
    C_D_std = drag_std / Q_times_A if Q_times_A != 0 else 0
    
    C_L_cov = (C_L_std / C_L * 100) if C_L != 0 else 0
    C_D_cov = (C_D_std / C_D * 100) if C_D != 0 else 0
    
    aoa_degrees = extract_aoa_number(aoa)
    config_parts = config.split('.')
    version_num = config_parts[POSITION_MAP['version']] if len(config_parts) > POSITION_MAP['version'] else 'unknown'
    
    coefficient_data[(config, aoa)] = {
        'config': config,
        'version_num': version_num,
        'aoa_degrees': aoa_degrees,
        'C_L': C_L,
        'C_D': C_D,
        'C_L_std': C_L_std,
        'C_D_std': C_D_std,
        'C_L_cov': C_L_cov,
        'C_D_cov': C_D_cov,
        'turbulence_model': data['turbulence_model'],
        'geometry': data['geometry'],
        'mesh': data['mesh'],
        'grid': data['grid']
    }

print(f"\n✓ Coefficients calculated for {len(coefficient_data)} configurations")

# Create dataframe
coeff_summary = []
for (config, aoa), coeff in sorted(coefficient_data.items(), 
                                    key=lambda x: (TURBULENCE_ORDER.get(x[1]['turbulence_model'], 99),
                                                  x[0][0],
                                                  x[1]['aoa_degrees'])):
    coeff_summary.append({
        'Configuration': config,
        'Turbulence Model': coeff['turbulence_model'],
        'AoA': aoa,
        'AoA (degrees)': coeff['aoa_degrees'],
        'Geometry': coeff['geometry'],
        'Grid': coeff['grid'],
        'C_L': f"{coeff['C_L']:.6f}",
        'C_L_std': f"{coeff['C_L_std']:.6f}",
        'C_L_COV (%)': f"{coeff['C_L_cov']:.1f}",
        'C_D': f"{coeff['C_D']:.6f}",
        'C_D_std': f"{coeff['C_D_std']:.6f}",
        'C_D_COV (%)': f"{coeff['C_D_cov']:.1f}"
    })

coeff_df = pd.DataFrame(coeff_summary)

# Save to Excel
with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    coeff_df.to_excel(writer, sheet_name='Coefficients', index=False)

print(f"✓ {len(coeff_summary)} coefficient rows saved to Excel")

# Apply formatting
time.sleep(0.5)
wb = load_workbook(excel_file)
ws = wb['Coefficients']

for column in ws.columns:
    max_length = 0
    column_letter = get_column_letter(column[0].column)
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 25)
    ws.column_dimensions[column_letter].width = adjusted_width

for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment
    cell.border = border_style

for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
    fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
    for cell in row:
        cell.alignment = data_alignment
        cell.border = border_style
        cell.fill = fill_color

ws.row_dimensions[1].height = 30
wb.save(excel_file)
wb.close()

print(f"✓ Coefficient data saved to Excel sheet: 'Coefficients'")


✓ Coefficients calculated for 16 configurations
✓ 16 coefficient rows saved to Excel
✓ Coefficient data saved to Excel sheet: 'Coefficients'
✓ Coefficient data saved to Excel sheet: 'Coefficients'


## Create Convergence Optimization Sheet (if convergence analysis was run)

In [17]:
if RUN_CONVERGENCE_ANALYSIS and convergence_results:
    print("\n" + "=" * 70)
    print("CREATING CONVERGENCE OPTIMIZATION SHEET")
    print("=" * 70)
    
    # Build optimized statistics comparison
    optimized_summary = []
    
    # Sort by turbulence model, then config, then AoA
    sorted_items = sorted(all_data.items(), 
                          key=lambda x: (TURBULENCE_ORDER.get(x[1]['turbulence_model'], 99),
                                        x[0][0],
                                        extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_items:
        if (config, aoa) in convergence_results:
            conv = convergence_results[(config, aoa)]
            
            # Find optimal trim points (minimum COV)
            lift_min_cov_idx = np.argmin(conv['lift']['cov'])
            drag_min_cov_idx = np.argmin(conv['drag']['cov'])
            
            optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
            optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
            
            # Use the more conservative (higher) trim value for both
            optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
            
            # Apply optimal trim
            lift_optimized = data['lift'][optimal_trim:]
            drag_optimized = data['drag'][optimal_trim:]
            
            # Calculate optimized statistics
            lift_mean_opt = np.mean(lift_optimized) if lift_optimized else 0
            lift_std_opt = np.std(lift_optimized) if lift_optimized else 0
            lift_cov_opt = (lift_std_opt / lift_mean_opt * 100) if lift_mean_opt != 0 else 0
            
            drag_mean_opt = np.mean(drag_optimized) if drag_optimized else 0
            drag_std_opt = np.std(drag_optimized) if drag_optimized else 0
            drag_cov_opt = (drag_std_opt / drag_mean_opt * 100) if drag_mean_opt != 0 else 0
            
            # Compare with original fixed-iteration statistics
            lift_original = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
            drag_original = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
            
            lift_mean_orig = np.mean(lift_original) if lift_original else 0
            lift_cov_orig = (np.std(lift_original) / lift_mean_orig * 100) if lift_mean_orig != 0 else 0
            
            drag_mean_orig = np.mean(drag_original) if drag_original else 0
            drag_cov_orig = (np.std(drag_original) / drag_mean_orig * 100) if drag_mean_orig != 0 else 0
            
            # Calculate improvements
            lift_cov_improvement = lift_cov_orig - lift_cov_opt
            drag_cov_improvement = drag_cov_orig - drag_cov_opt
            
            optimized_summary.append({
                'Turbulence Model': data['turbulence_model'],
                'Configuration': config,
                'AoA': aoa,
                'Original Iterations': len(lift_original),
                'Optimal Trim': optimal_trim,
                'Optimized Iterations': len(lift_optimized),
                'Lift Mean (Orig)': f"{lift_mean_orig:.3f}",
                'Lift Mean (Opt)': f"{lift_mean_opt:.3f}",
                'Lift COV (Orig)': f"{lift_cov_orig:.2f}%",
                'Lift COV (Opt)': f"{lift_cov_opt:.2f}%",
                'Lift COV Δ': f"{lift_cov_improvement:+.2f}%",
                'Drag Mean (Orig)': f"{drag_mean_orig:.3f}",
                'Drag Mean (Opt)': f"{drag_mean_opt:.3f}",
                'Drag COV (Orig)': f"{drag_cov_orig:.2f}%",
                'Drag COV (Opt)': f"{drag_cov_opt:.2f}%",
                'Drag COV Δ': f"{drag_cov_improvement:+.2f}%"
            })
    
    # Save optimized statistics to Excel
    optimized_df = pd.DataFrame(optimized_summary)
    
    with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        optimized_df.to_excel(writer, sheet_name='Optimized_Statistics', index=False)
    
    # Apply formatting
    time.sleep(0.5)
    wb = load_workbook(excel_file)
    ws = wb['Optimized_Statistics']
    
    for column in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 25)
        ws.column_dimensions[column_letter].width = adjusted_width
    
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        for cell in row:
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
    
    ws.row_dimensions[1].height = 30
    wb.save(excel_file)
    wb.close()
    
    print(f"\n✓ Optimized statistics sheet created with {len(optimized_summary)} rows")
    print(f"✓ Compares original vs optimized convergence trim for all configurations")
else:
    print("\n⚠ Skipping Optimized_Statistics sheet (convergence analysis not run)")


CREATING CONVERGENCE OPTIMIZATION SHEET

✓ Optimized statistics sheet created with 16 rows
✓ Compares original vs optimized convergence trim for all configurations

✓ Optimized statistics sheet created with 16 rows
✓ Compares original vs optimized convergence trim for all configurations


## Summary

In [18]:
print("\n" + "=" * 70)
print("EXCEL GENERATION COMPLETE")
print("=" * 70)
print(f"\nFile: {excel_file}")
print("\nSheets created:")
print("  1. Data Summary - Grouped by configuration")
print("  2. Turbulence_Comparison - 4 tables (Lift, Drag, Lift COV, Drag COV)")
print("  3. Coefficients - C_L and C_D with COV")
if RUN_CONVERGENCE_ANALYSIS and convergence_results:
    print("  4. Optimized_Statistics - Convergence optimization comparison")
print("\n✓ All Excel outputs generated successfully!")


EXCEL GENERATION COMPLETE

File: C:\Users\lukek\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\SUMMARY_Statistics.xlsx

Sheets created:
  1. Data Summary - Grouped by configuration
  2. Turbulence_Comparison - 4 tables (Lift, Drag, Lift COV, Drag COV)
  3. Coefficients - C_L and C_D with COV
  4. Optimized_Statistics - Convergence optimization comparison

✓ All Excel outputs generated successfully!
